# Análisis de outliers y correlaciones — Producción (Fase 3, ML)

**Rol 1 — ML Engineer.** Continúa el EDA (`eda_produccion.ipynb`) con foco en:
1. **Outliers y distribución del target** → respalda la decisión de transformar con `log1p` y usar **MAE** (robusta).
2. **Anomalías** en otras medidas (valores imposibles).
3. **Matriz de correlación** entre medidas numéricas.
4. **Preview de features autoregresivas** (lags, media móvil, antigüedad) vs el target del mes siguiente.

Las conclusiones alimentan el **ADR-028** (diseño/validación) y la definición de features con el Rol 2.

> Las features finales las materializa el Rol 2 en el feature store; acá las construimos al vuelo solo para **validar que sirven**.

## 0. Setup (universo petrolero)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV = "../data/_explore/produccion_full.csv"
num = ["prod_pet","prod_gas","prod_agua","iny_agua","iny_gas","iny_co2","iny_otro","tef","profundidad"]
cols = ["idpozo","anio","mes","cuenca","formacion","tipopozo"] + num
df = pd.read_csv(CSV, usecols=cols, encoding="utf-8-sig")
for c in num:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["periodo"] = pd.to_datetime(dict(year=df.anio, month=df.mes, day=1))

# universo petrolero: pozos con al menos un mes de prod_pet>0
pozos_pet = df.loc[df.prod_pet > 0, "idpozo"].unique()
dfp = df[df.idpozo.isin(pozos_pet)].copy()
print(f"Filas (universo petrolero): {len(dfp):,} | pozos: {dfp.idpozo.nunique():,}")

## 1. Outliers y distribución del target `prod_pet`

Buscamos cuán pesada es la cola y cuántos valores extremos hay.

In [ ]:
pet = dfp.loc[dfp.prod_pet > 0, "prod_pet"]
print("Percentiles de prod_pet (>0):")
print(pet.quantile([.5,.75,.90,.95,.99,.999,1.0]).round(1))

# Regla IQR para contar outliers
q1, q3 = pet.quantile([.25,.75]); iqr = q3 - q1
techo = q3 + 1.5*iqr
n_out = (pet > techo).sum()
print(f"\nTecho IQR (Q3+1.5·IQR): {techo:.1f}")
print(f"Outliers por IQR: {n_out:,} ({100*n_out/len(pet):.1f}% de los meses con producción)")

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].boxplot(pet, vert=True, showfliers=True)
ax[0].set_title("Boxplot prod_pet (>0) — cola larguísima"); ax[0].set_ylabel("m³")
ax[1].boxplot(np.log1p(pet), vert=True)
ax[1].set_title("Boxplot log1p(prod_pet) — escala comprimida")
plt.show()

## 2. Target crudo vs `log1p` (justifica la transformación)

Si el `log1p` reduce mucho la asimetría (skewness), conviene modelar sobre el target transformado.

In [ ]:
sk_raw = pet.skew()
sk_log = np.log1p(pet).skew()
print(f"Skewness prod_pet crudo : {sk_raw:.2f}  (muy asimétrica si >> 0)")
print(f"Skewness log1p(prod_pet): {sk_log:.2f}  (cercana a 0 = casi simétrica)")

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(pet, bins=80); ax[0].set_title(f"prod_pet crudo (skew={sk_raw:.1f})")
ax[1].hist(np.log1p(pet), bins=80); ax[1].set_title(f"log1p(prod_pet) (skew={sk_log:.1f})")
plt.show()

## 3. Anomalías en otras medidas

Chequeo de valores imposibles / sospechosos que habría que tratar antes de entrenar.

In [ ]:
# tef = tiempo efectivo del mes: en días NO debería superar ~31
print("tef: rango", (dfp.tef.min(), round(dfp.tef.max(),1)), "| media", round(dfp.tef.mean(),1))
print(f"  meses con tef > 31 (sospechoso): {(dfp.tef > 31).sum():,} ({100*(dfp.tef>31).mean():.2f}%)")

# valores negativos en medidas (imposibles)
for c in ["prod_pet","prod_gas","prod_agua","tef","profundidad"]:
    neg = (dfp[c] < 0).sum()
    if neg: print(f"  {c}: {neg:,} valores negativos (!)")

# % de ceros por medida
print("\n% de ceros por medida:")
print((dfp[["prod_pet","prod_gas","prod_agua","tef"]] == 0).mean().mul(100).round(1))

## 4. Matriz de correlación entre medidas numéricas

¿Qué medidas se mueven juntas? Útil para detectar redundancia y relaciones (ej. inyección ↔ producción).

In [ ]:
num_cols = ["prod_pet","prod_gas","prod_agua","iny_agua","iny_gas","tef","profundidad"]
corr = dfp[num_cols].corr()

fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=45, ha="right")
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Matriz de correlación (medidas numéricas)")
plt.tight_layout(); plt.show()
corr.round(2)

## 5. Preview de features autoregresivas vs target (t+1)

Construimos al vuelo, **por pozo y ordenado por tiempo**, las features candidatas y medimos su correlación
con el target = `prod_pet` del **mes siguiente**. Sirve para validar el feature engineering antes de
materializarlo en el feature store.

⚠️ Todas las features usan **solo información hasta el mes actual** (anti-leakage).

In [ ]:
d = dfp.sort_values(["idpozo","periodo"]).copy()
g = d.groupby("idpozo")["prod_pet"]
d["target_next"] = g.shift(-1)                # prod_pet[t+1]  <- lo que queremos predecir
d["pet_t"]       = d["prod_pet"]              # valor actual (persistencia)
d["lag1"]        = g.shift(1)                 # t-1
d["lag2"]        = g.shift(2)                 # t-2
d["roll3"]       = g.transform(lambda s: s.rolling(3).mean())  # media móvil 3 meses
d["antiguedad"]  = d.groupby("idpozo").cumcount()             # meses desde el 1er registro

feats = ["pet_t","lag1","lag2","roll3","antiguedad","tef"]
corr_t = d[feats + ["target_next"]].dropna().corr()["target_next"].drop("target_next")
print("Correlación de cada feature candidata con el target (prod_pet del mes siguiente):")
print(corr_t.sort_values(ascending=False).round(3))

corr_t.sort_values().plot(kind="barh", figsize=(7,3.5),
    title="Correlación feature ↔ target (t+1)")
plt.xlabel("correlación de Pearson"); plt.tight_layout(); plt.show()

## 6. Target por categóricas

¿La producción difiere por cuenca / formación? Si sí, son buenas features categóricas.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
dfp.groupby("cuenca").prod_pet.median().sort_values().plot(kind="barh", ax=ax[0],
    title="prod_pet mediana por cuenca")
(dfp.groupby("formacion").prod_pet.median().sort_values(ascending=False).head(10)
   .plot(kind="barh", ax=ax[1], title="prod_pet mediana — top 10 formaciones"))
plt.tight_layout(); plt.show()

## 7. Conclusiones → decisiones para el ADR / feature store

1. **Target muy sesgado con outliers fuertes** → confirmar `log1p` como target y **MAE** como métrica principal (robusta).
2. **Anomalías a tratar** en limpieza: `tef > 31` (imposible en días), revisar antes de usar `tef` como feature.
3. **Correlaciones entre medidas** → detectar redundancias (no meter features que digan lo mismo).
4. **Features autoregresivas validadas:** las de mayor correlación con el target son las que el feature store debe priorizar.
5. **Categóricas con señal** (cuenca, formación) → incluirlas como features.

Esto se coordina con el Rol 2 para definir el esquema del feature store.